# Vaccine Tracker

- The first dataset is the 2019 US Census that has the total number of inhabitants of each state, along with latitude and longitude data for each state's capital city.
- The second dataset contains an estimate for the total number of people that have been vaccinated in each state at the time.

In the next code cell, we load and preprocess the data.  As output, we'll see the total percent of the population that has been vaccinated in the US, along with a preview of the Pandas DataFrame that we'll use to make the tracker.

In [ ]:
# Imports
import pandas as pd
from datetime import date
import folium
from folium import Marker
from folium.plugins import MarkerCluster
import math
import matplotlib.pyplot as plt
import seaborn as sns

# Population Data
populationData = pd.read_csv('2019_Census_US_Population_Data_By_State_Lat_Long.csv')

# Vaccination data for the latest available date in the dataset
vaccinationData = pd.read_csv('us_state_vaccinations.csv')
vaccinationData['date'] = pd.to_datetime(vaccinationData['date'])

latestDate = vaccinationData['date'].max()
latestDateStr = latestDate.strftime('%Y-%m-%d')

vaccinationByLocation = vaccinationData.loc[
    vaccinationData['date'] == latestDate,
    ["location", "people_vaccinated"]
]

# Vaccination and population data
vaccinationAndPopulationByLocation = pd.merge(
    populationData,
    vaccinationByLocation,
    left_on='STATE',
    right_on='location'
).drop(columns='location')

# Calculate percentage vaccinated by state
vaccinationAndPopulationByLocation['percent_vaccinated'] = (
    vaccinationAndPopulationByLocation['people_vaccinated'] /
    vaccinationAndPopulationByLocation['POPESTIMATE2019']
)

print('Data date:', latestDateStr)
vaccinationAndPopulationByLocation

,STATE,POPESTIMATE2019,lat,long,people_vaccinated,percent_vaccinated


In [ ]:
print("Date ran:", date.today())

# Calculate the total percent vaccinated in the US
percentageTotal = vaccinationAndPopulationByLocation["people_vaccinated"].sum() / vaccinationAndPopulationByLocation["POPESTIMATE2019"].sum()
print('Percentage Vaccinated in the US: {}%'.format(round(percentageTotal*100, 2))) 

The next code cell uses the data to create a tracker, with one marker for each state.  We can click on the markers to see the percentage of the population that has been vaccinated.

In [ ]:
# Create the map
v_map = folium.Map(location=[42.32,-71.0589], tiles='cartodbpositron', zoom_start=4) 

# Add points to the map
mc = MarkerCluster()
for idx, row in vaccinationAndPopulationByLocation.iterrows(): 
    if not math.isnan(row['long']) and not math.isnan(row['lat']):
        mc.add_child(Marker(location=[row['lat'], row['long']],
                            tooltip=str(round(row['percent_vaccinated']*100, 2))+"%"))
v_map.add_child(mc)

# Display the map
v_map